<a href="https://colab.research.google.com/github/CrislaniaAlmeida/Sentinel_Ai/blob/main/sentinel_ia_deteccao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/CrislaniaAlmeida/Sentinel_Ai.git
!ls Sentinel_Ai/frontend/video/cameras

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolo11n.pt")

resultados = modelo.predict(
    source="Sentinel_Ai/frontend/video/cameras/cam-07.mp4",
    classes=[0, 2],
    conf=0.35,
    save=True,
    project="resultados",
    name="cam-07",
    exist_ok=True,
    verbose=False,
)

print("Quadros analisados:", len(resultados))
print("Pessoas no primeiro quadro:", int((resultados[0].boxes.cls == 0).sum()))
print("Carros no primeiro quadro:", int((resultados[0].boxes.cls == 2).sum()))

In [ ]:
import glob, subprocess
from base64 import b64encode
from IPython.display import HTML

arquivo = glob.glob("runs/detect/resultados/cam-07/*")[0]
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", arquivo,
                "-vcodec", "libx264", "-pix_fmt", "yuv420p", "resultado_cam07.mp4"])

video = b64encode(open("resultado_cam07.mp4", "rb").read()).decode()
HTML(f'<video width="800" controls autoplay loop muted><source src="data:video/mp4;base64,{video}" type="video/mp4"></video>')

In [ ]:
rastros = {}

resultados = modelo.track(
    source="Sentinel_Ai/frontend/video/cameras/cam-07.mp4",
    classes=[0, 2],
    conf=0.35,
    tracker="bytetrack.yaml",
    persist=False,
    stream=True,
    save=True,
    line_width=1,
    project="rastreamento",
    name="cam-07",
    exist_ok=True,
    verbose=False,
)

for numero_quadro, r in enumerate(resultados):
    if r.boxes.id is None:
        continue
    for id_rastro, classe, conf in zip(r.boxes.id.int().tolist(),
                                       r.boxes.cls.int().tolist(),
                                       r.boxes.conf.tolist()):
        segundo = numero_quadro / 24
        tipo = "pessoa" if classe == 0 else "carro"
        chave = (id_rastro, tipo)
        if chave not in rastros:
            rastros[chave] = {"entrada": segundo, "saida": segundo, "conf_max": conf}
        else:
            rastros[chave]["saida"] = segundo
            rastros[chave]["conf_max"] = max(rastros[chave]["conf_max"], conf)

confiaveis = {k: d for k, d in rastros.items()
              if d["saida"] - d["entrada"] >= 1.0 and d["conf_max"] >= 0.50}

print(f"Rastros brutos: {len(rastros)}  |  Rastros confiáveis: {len(confiaveis)}\n")
for (i, tipo), d in sorted(confiaveis.items()):
    print(f"id {i:>3}  {tipo:<7} de {d['entrada']:5.1f}s até {d['saida']:5.1f}s"
          f"  (confiança máx. {d['conf_max']:.0%})")

In [ ]:
import glob, subprocess
from base64 import b64encode
from IPython.display import HTML

arquivo = glob.glob("runs/detect/rastreamento/cam-07/*")[0]
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", arquivo,
                "-vcodec", "libx264", "-pix_fmt", "yuv420p", "rastreamento_cam07.mp4"])

video = b64encode(open("rastreamento_cam07.mp4", "rb").read()).decode()
HTML(f'<video width="800" controls autoplay loop muted><source src="data:video/mp4;base64,{video}" type="video/mp4"></video>')

In [ ]:
quadros_com_pessoa = []
maior_conf = 0

for numero_quadro, r in enumerate(modelo.predict(
        source="Sentinel_Ai/frontend/video/cameras/cam-07.mp4",
        classes=[0], conf=0.35, stream=True, verbose=False)):
    if len(r.boxes) > 0:
        quadros_com_pessoa.append(numero_quadro)
        maior_conf = max(maior_conf, float(r.boxes.conf.max()))

print("Quadros com pessoa:", len(quadros_com_pessoa), "de 456")
print("Maior confiança:", f"{maior_conf:.0%}")
print("Primeiros quadros onde aparece:", quadros_com_pessoa[:30])

In [ ]:
!pip install -q psycopg2-binary

from google.colab import userdata
from sqlalchemy import create_engine, text

engine = create_engine(userdata.get("DATABASE_URL"))

with engine.connect() as conexao:
    tenant_id = conexao.execute(
        text("SELECT tenant_id FROM usuarios WHERE email = :email"),
        {"email": "crislania@sentinel.ai"},
    ).scalar_one()
    total_cameras = conexao.execute(
        text("SELECT count(*) FROM cameras WHERE tenant_id = :t"),
        {"t": tenant_id},
    ).scalar_one()

print("Conectado ao Neon! ✅")
print("Tenant:", tenant_id)
print("Câmeras cadastradas hoje:", total_cameras)

In [ ]:
CAMERAS_SHOPPING = [
    ("cam-01.mp4", "CAM-01 · Entrada Norte",              "Piso L1 · Acesso Norte", "ONLINE"),
    ("cam-02.mp4", "CAM-02 · Átrio Central",              "Piso L1 · Centro",       "ONLINE"),
    ("cam-03.mp4", "CAM-03 · Praça de Alimentação Norte", "Piso L2 · Ala Norte",    "ONLINE"),
    ("cam-04.mp4", "CAM-04 · Corredor Oeste",             "Piso L1 · Ala Oeste",    "ONLINE"),
    ("cam-05.mp4", "CAM-05 · Escada Rolante",             "Piso L1 → L2",           "ONLINE"),
    ("cam-06.mp4", "CAM-06 · Foyer do Cinema",            "Piso L3",                "ONLINE"),
    ("cam-07.mp4", "CAM-07 · Estacionamento G2 Sul",      "Subsolo G2",             "ONLINE"),
    ("cam-08.mp4", "CAM-08 · Corredor Norte",             "Piso L2 · Ala Norte",    "OFFLINE"),
    ("cam-09.mp4", "CAM-09 · Acesso aos Sanitários",      "Piso L1 · Ala Leste",    "MANUTENCAO"),
    ("cam-10.mp4", "CAM-10 · Entrada Principal Sul",      "Piso L1 · Acesso Sul",   "ONLINE"),
    ("cam-11.mp4", "CAM-11 · Praça de Alimentação Sul",   "Piso L2 · Ala Sul",      "ONLINE"),
    ("cam-12.mp4", "CAM-12 · Área de Serviço",            "Elevador de Carga · L1", "ONLINE"),
]

cameras_para_processar = []

with engine.begin() as conexao:
    for video, nome, local, status in CAMERAS_SHOPPING:
        endereco = f"arquivo://{video}"
        camera_id = conexao.execute(
            text("SELECT id FROM cameras WHERE tenant_id = :t AND rtsp_url = :u"),
            {"t": tenant_id, "u": endereco},
        ).scalar()

        if camera_id is None:
            camera_id = conexao.execute(
                text("""INSERT INTO cameras (id, tenant_id, nome, local_instalacao, rtsp_url, status, criado_em)
                        VALUES (gen_random_uuid(), :t, :n, :l, :u, :s, now())
                        RETURNING id"""),
                {"t": tenant_id, "n": nome, "l": local, "u": endereco, "s": status},
            ).scalar_one()
            acao = "cadastrada"
        else:
            conexao.execute(
                text("UPDATE cameras SET nome = :n, local_instalacao = :l, status = :s WHERE id = :id"),
                {"n": nome, "l": local, "s": status, "id": camera_id},
            )
            acao = "já existia"

        print(f"{nome:<38} {status:<11} {acao}")
        if status == "ONLINE":
            cameras_para_processar.append((str(camera_id), nome, video))

print(f"\nCâmeras que serão processadas: {len(cameras_para_processar)}")

In [ ]:
def processar_camera(caminho_video):
    rastros = {}

    for numero_quadro, r in enumerate(modelo.track(
            source=caminho_video, classes=[0, 2], conf=0.35,
            tracker="bytetrack.yaml", persist=False, stream=True, verbose=False)):
        if r.boxes.id is None:
            continue
        segundo = numero_quadro / 24

        for id_rastro, classe, conf, caixa in zip(r.boxes.id.int().tolist(),
                                                  r.boxes.cls.int().tolist(),
                                                  r.boxes.conf.tolist(),
                                                  r.boxes.xyxy.tolist()):
            tipo = "PESSOA" if classe == 0 else "VEICULO"
            chave = (id_rastro, tipo)
            if chave not in rastros:
                rastros[chave] = {"entrada": segundo, "saida": segundo, "confs": [], "amostras": {}}
            ficha = rastros[chave]
            ficha["saida"] = segundo
            ficha["confs"].append(conf)

            segundo_inteiro = int(segundo)
            if segundo_inteiro not in ficha["amostras"]:
                ficha["amostras"][segundo_inteiro] = {
                    "segundo": round(segundo, 2),
                    "conf": conf,
                    "caixa": [round(v, 1) for v in caixa],
                }

    return {chave: ficha for chave, ficha in rastros.items()
            if ficha["saida"] - ficha["entrada"] >= 1.0 and max(ficha["confs"]) >= 0.50}

print("Função processar_camera pronta ✅")

In [ ]:
import os

!pip install -q ultralytics

if not os.path.exists("Sentinel_Ai"):
    !git clone -q https://github.com/CrislaniaAlmeida/Sentinel_Ai.git

from ultralytics import YOLO
modelo = YOLO("yolo11n.pt")
print("Modelo carregado ✅")

In [ ]:
engine = create_engine(
    userdata.get("DATABASE_URL"),
    pool_pre_ping=True,
    pool_recycle=300,
)
print("Conexão recriada com verificação automática ✅")

In [ ]:
import json, time, uuid
from datetime import datetime, timedelta

inicio = datetime.utcnow()
total_eventos = total_deteccoes = 0

for camera_id, nome, video in cameras_para_processar:
    t0 = time.time()
    rastros = processar_camera(f"Sentinel_Ai/frontend/video/cameras/{video}")

    eventos, deteccoes = [], []
    for (id_rastro, tipo), ficha in rastros.items():
        evento_id = str(uuid.uuid4())
        eventos.append({
            "id": evento_id, "t": tenant_id, "c": camera_id,
            "tipo": "PESSOA_DETECTADA" if tipo == "PESSOA" else "VEICULO_DETECTADO",
            "ts": inicio + timedelta(seconds=ficha["entrada"]),
            "conf": sum(ficha["confs"]) / len(ficha["confs"]),
        })
        for amostra in ficha["amostras"].values():
            x1, y1, x2, y2 = amostra["caixa"]
            deteccoes.append({
                "id": str(uuid.uuid4()), "t": tenant_id, "e": evento_id, "c": camera_id,
                "tipo": tipo, "conf": amostra["conf"],
                "ts": inicio + timedelta(seconds=amostra["segundo"]),
                "bbox": json.dumps({"x1": x1, "y1": y1, "x2": x2, "y2": y2,
                                    "rastro": id_rastro, "segundo_video": amostra["segundo"]}),
            })

    with engine.begin() as conexao:
        conexao.execute(text("DELETE FROM deteccoes WHERE camera_id = :c"), {"c": camera_id})
        conexao.execute(text("DELETE FROM eventos WHERE camera_id = :c"), {"c": camera_id})
        if eventos:
            conexao.execute(text(
                """INSERT INTO eventos (id, tenant_id, camera_id, tipo_evento, timestamp, confianca_media)
                   VALUES (:id, :t, :c, :tipo, :ts, :conf)"""), eventos)
        if deteccoes:
            conexao.execute(text(
                """INSERT INTO deteccoes (id, tenant_id, evento_id, camera_id, tipo, bbox_json, confianca, timestamp)
                   VALUES (:id, :t, :e, :c, :tipo, :bbox, :conf, :ts)"""), deteccoes)

    pessoas = sum(1 for (_, tipo) in rastros if tipo == "PESSOA")
    veiculos = len(rastros) - pessoas
    total_eventos += len(eventos)
    total_deteccoes += len(deteccoes)
    print(f"{nome:<38} {pessoas:>3} pessoas  {veiculos:>3} veículos  "
          f"{len(deteccoes):>4} detecções  ({time.time() - t0:.0f}s)")

print(f"\nTotal gravado no Neon: {total_eventos} eventos e {total_deteccoes} detecções ✅")